# 性能调优——数据驱动的迭代闭环

> **02.02 · 建议时长 ~40 min · 前置要求：02.01（或同等经验）**

下一节先看一个反直觉现象：多算 1.5 万倍数据，耗时不变。正确只是及格线，接下来要回答：**它跑得快吗？慢在哪？还能快多少？** 本节不再复述流程，而是讲**调优的思维方式**与**在自己的算子上怎么用**，附 16 个真实网络用例的精编证据（Ascend 910B3，NPU event 计时，warm-up 3 + 10 次取中位）。

本节实录是一次**诚实的调优**：一轮优化收益 1~3%，一个大方向测算后决定走需求变更，没有漂亮的全线翻倍，数据说什么就是什么。

路线：**三个设计思想 → 行动模板 → 实录精编 → 演进与停止**。全程约 40 分钟；课后实践另计。



## 26 万元素和 17 元素，耗时一模一样

02.01 节交付的 xlog1py 正确了（40/40），但真机采集出一个奇怪的现象：**26 万元素的用例和 17 元素的用例，耗时一模一样**（都是 ~0.18 ms）。多算 1.5 万倍的数据，时间却没变：瓶颈根本不在计算。

正确只是及格线。**它跑得快吗？慢在哪？还能快多少？** 本节讲数据驱动的调优思维方式与行动模板。

## 1. 调优的三个设计思想

<img src="./images/02.02_perf_loop.png" alt="数据驱动的性能调优闭环与三档读法" width="700px">

**思想一：数据先于观点。** 闭环五步（采集 → 定位 → 策略 → 实施 → 回归）的每一步都在防同一件事：凭感觉判断快慢。体感会骗人：本节实录中，26 万元素与 17 元素的用例耗时几乎相同；凭直觉会去优化 kernel，而数据表明瓶颈根本不在算子层。

**思想二：分档读法。** 拿到逐用例数据先分档，不同规模的**开销结构完全不同，看的指标也不同**：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">规模档</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">看什么</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">判定信号</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">L（≥1 亿元素）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">有效带宽对 HBM 峰值的比值</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">~70% 即**访存受限**：算力不是瓶颈，搬运是</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">M / S（更小）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">是否在 launch 地板</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">耗时与数据量无关的 ~0.2ms 定值：kernel 本体 <10μs，调度开销主导</td></tr></tbody></table>

**档位判断错，后面全白干**：在 launch 地板上优化 kernel 指令，一微秒也赚不到。

**思想三：流量思维。** 算子层性能的大头往往不是算得慢，而是搬得多。一个指标就能暴露搬运浪费：**y_compact = 原始数据量 / 物化后数据量**，它把需求期的便利决定翻译成性能期的带宽账单（本节实录：最高放大 **866 万倍**）。

## 2. 在你的算子上怎么用

行动模板（依次回答五个问题）：

1. **采集**：warm-up + 多次取中位 + 逐用例记录（方法学比单个数字重要：可复现、抗抖动）；
2. **判读**：先分档，再按档看对应指标，写出一句瓶颈判定（如 L 档访存受限、S 档 launch 地板）；
3. **一轮优化**：选与瓶颈类型匹配的手段（访存受限 → 减流量 / 调搬运粒度，不必抠指令；地板 → 算子层无空间，考虑融合/批处理）；
4. **回归**：精度全量通过才可保留；逐用例看加速比，防 L 档涨、S 档跌；
5. **停止判断**：见第 4 节的三个信号。

工具链（cannbot-skills 仓 `ops/` 域 Skill 组合）：`ops-profiling`（采集）、`ascendc-perf-optimize`（定位）、`ascendc-performance-best-practices`（按算子族的策略库）。注：这三个 Skill 均在 [cann/cannbot-skills](https://gitcode.com/cann/cannbot-skills) 仓 `ops/` 域，克隆仓库后 Agent 可按需加载。培训课件中的旧插件名 ops-perf-optimize 已由上述 Skill 组合替代。

## 3. 实录精编：16 用例体检

**基线（代表档位，共 16 用例真机实测）**：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">用例</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">元素数</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">档</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">耗时</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">有效带宽</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">xlog1py_47</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">8.7e7</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">L</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">0.935 ms</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">1112 GB/s</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">xlog1py_179</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">2.25e8</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">L</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">2.55 ms</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">1060 GB/s</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">xlog1py_22</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">2.6e6</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">M</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">0.18 ms</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">173 GB/s</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">xlog1py_59</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">7e4</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">S</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">0.19 ms</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">4.5 GB/s</td></tr></tbody></table>

**判读**：L 档带宽达 HBM 峰值 ~70% → **访存受限**；M/S 档全部落在 ~0.18ms 定值 → **launch 开销地板**。

**流量发现（本节最重要的洞察）**：16 个用例中 11 个的 Y 是广播输入，y_compact 最小 **26 / 2.25e8**，02.01 节启动前问卷拍板等形输入（host 物化广播）时的便利，在这里变成了最高 **866 万倍**的 GM 读流量放大。**规格也是性能变量**：需求期的每个取舍都可能在性能期付账。

**一轮优化（诚实版）**：针对访存受限调大搬运粒度（tileSize 4096→5120），实测 L 档 **−1.8~3.3%**、S 档不动、精度 40/40 无损，保留。收益递减：tileSize 已到 UB 物理上限，带宽余量靠常规 tiling 已挤不出。**小收益不丢人，假装有大收益才丢人。**

## 4. 方向演进与停止条件

剩下的大头是流量：**in-kernel broadcast**（只读 Y 的紧凑原始数据，在 UB 内展开，模板来自 ascendc-performance-best-practices 广播原语）可省 L 档约一半读流量。参考实现实测（dav-3510）：**L 档 1.34~2.66×，但 S 档退化到 0.56~0.82×**，没有免费的午餐，必须按 y_compact 与规模**分型分派**（L 档走 in-kernel，S 档保留物化）。

但这条路撞上了 02.01 节拍板的等形输入规格。**正确的动作不是绕过需求偷偷改 kernel，而是拿数据回到需求变更**（y_compact 与流量测算就是最好的论据）。调优发现规格的代价，数据推动需求演进，这是数据驱动开发的完整形态。

**三个停止信号**（任一出现即收口）：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">信号</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">含义</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">达到验收口径</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">目标完成，继续优化是镀金</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">收益递减</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">本轮 1~3% 且已到物理上限，下一轮预期更低</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">下一步需要变更需求</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">那是另一个决策循环，不是本算子调优的延续</td></tr></tbody></table>

**经验精选**（实测得出，可直接迁移）：三档读法是第一入口；带宽 ~70% 即访存受限、别再抠指令；优化必须逐用例验证（L 涨 S 跌是事故）；广播物化的代价要在需求期用 y_compact 量化。

## 5. 带走物

| 类别 | 内容 |
|---|---|
| **一句话** | 性能优化是数据驱动的科学实验，不是碰运气 |
| **工具** | 三原则（数据先于观点 / 分档读法 / 流量思维）+ y_compact 指标 |
| **模板** | 调优行动模板：采集 → 瓶颈判定 → 一轮优化 → 回归 → 停止判断 |
| **下一步** | [02.03](./02.03_gitcode_collaboration.ipynb)：代码写对了、跑快了，怎么进仓库 |

## 6. 课后练习

> 练习答案：[`answer/02.02_answer.txt`](./answer/02.02_answer.txt)



> 💡 **项目线第 2 站**：优先沿用[02.01](./02.01_complete_workflow.ipynb)实践题开发的算子，下一节将为它走完提交进仓库的流程。


1. （单选题）某算子 2.25 亿元素用例耗时 2.55 ms、有效带宽 1060 GB/s（HBM 峰值约 1.6 TB/s），最可能的瓶颈判定是？  
    A. 计算受限：指令链太长  
    B. 访存受限：带宽已达混合负载的健康区间  
    C. 同步受限：流水线中断  
    D. launch 开销地板

2. （单选题）26 万元素与 17 元素两个用例耗时都是 ~0.18 ms，说明什么？  
    A. 测量误差，需要增加重复次数  
    B. 小数据被 launch/调度开销地板主导，算子层优化无空间  
    C. kernel 写错了，结果不可信  
    D. 应该减少 numBlocks

3. （多选题）关于优化后必须逐用例验证，参考实现的 in-kernel broadcast 方案说明了什么？  
    A. L 档 1.34~2.66× 的收益不能外推到全部用例  
    B. S 档退化 0.56~0.82×，全量切换会造成性能事故  
    C. 应按 y_compact 与规模分型分派  
    D. 加速比数据只需要在最大的用例上测一次

4. （编程实践·必做）对你自己的算子（上一节作业或任意已有实现）执行一次完整调优闭环：不少于 6 个用例的基线 → 三档读法瓶颈判定 → 至少一轮优化 → 精度回归 → 逐用例加速比表与停止/继续结论。**另附一段数据反思：哪一步的数据改变了你最初的观点？** 若优化方向需要变更需求，写出需求演进建议及数据依据。

5. （编程实践·选做）需求演进实操：为本节的 xlog1py（或自己的含广播输入算子）起草一份 in-kernel broadcast 需求演进提案，包含 y_compact 数据论据、收益测算（对照参考实现 L 档 1.34~2.66× / S 档退化）、分型分派方案、可直接发给 PM 的需求变更描述（五要素齐全）。

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/02.02_answer.txt

## 7. 要点回顾

- **三个设计思想**：数据先于观点（闭环每步防主观臆断）、分档读法（档位判断错全白干）、流量思维（y_compact 把规格代价翻译成带宽账单）；
- **行动模板**：采集方法学 → 一句瓶颈判定 → 与瓶颈匹配的一轮优化 → 回归（防 L 涨 S 跌）→ 停止判断；
- **实录证据**：L 档访存受限（~70% 峰值）、M/S 档 launch 地板、广播物化放大 866 万倍、一轮优化诚实收益 1~3%；
- **演进与停止**：in-kernel broadcast 需分型分派且需需求变更，数据推动需求演进；三个停止信号。

三原则是在 **3 步链**的 xlog1py 上总结的。留一个问题给进阶路径：如果一个算子每搬运 1 字节要执行 **4 倍**的指令，访存受限还成立吗？[进阶路径](../README.md)用 gelu_approx 重验这个假设。最后一节：[协作上库与边界——课程收束](./02.03_gitcode_collaboration.ipynb)。